
- **Separate `m_inf` and `h_inf`.** Each segment used one `x_inf` for both gates.
- **All 10 segments fitted at once.** One shared `e_r` and `g_bar`; segments at the same voltage share one set of `tau_m`, `tau_h`, `m_inf`, `h_inf`. 
- **m is always the fast gate.** `tau_h` is fitted as a ratio ≥ 1 to `tau_m`; otherwise the fit can swap m and h between segments, since the current only uses `m·h`.
- **Gates rescaled to peak at 1**, with the factor moved into `g_bar`.
- **Gates start at their −60 mV steady state**, not `m = 0`, `h = 1`.
- **Whole-trace model uses the fitted curves.** Sigmoids for `m_inf`/`h_inf` and Gaussians for `tau_m`/`tau_h` replace the `np.interp` tables (flat outside −100 to +20 mV) and the single `tau_m`/`tau_h`. 12 parameters instead of 16.
- **The segment curves are start 0** of the whole-trace fit; the other 19 starts are random.
- **Bounds are used.** `lower`/`upper` were defined but never passed to a fit.
- **Gaussian fits start wider** (width 2000 instead of 5, which failed to converge).
- **Best start refitted** with up to 200 evaluations (was stopped at 40).

In [ ]:
# %% SECTION 1: Packages import
import os
import numpy as np

# Import other packages as necessary
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

In [ ]:
# %% SECTION 2: Helper functions
# Define functions if needed
# Bounds: [e_r, g_bar, m_inf steepness, m_inf center, h_inf steepness, h_inf center, tau_m scale, center, width, tau_h scale, center, width]
lower = np.array([-150, 0,   0, -150, -1, -150, 0.01, -200, 1,   0.01, -200, 1  ])
upper = np.array([ 150, 100, 1,  150,  0,  150, 1e3,   200, 1e5, 1e4,   200, 1e5])
rng = np.random.default_rng(0)   # fixed seed -> reproducible
n_starts = 20

def gate_model(tau, x_0,x_inf,dt):
    y = x_0 + (x_inf - x_0) * (1 - np.exp(-dt / tau))
    return y

# Per-segment model: one voltage segment, its own m_inf and h_inf, carries m/h between segments
def get_current_segment(e_r, g_bar, tau_m, tau_h, m_inf, h_inf, v, t, m_0, h_0):
    m = np.zeros(len(v))
    h = np.zeros(len(v))

    m[0] = m_0
    h[0] = h_0

    for i in range(1, len(t)):
        dt = t[i] - t[i-1]

        m[i] = gate_model(tau_m, m[i-1], m_inf, dt)
        h[i] = gate_model(tau_h, h[i-1], h_inf, dt)

    return g_bar * (v - e_r) * m * h, m[-1], h[-1]

# [tau_m, tau_h, m_inf, h_inf] at step voltage number k. tau_h is stored as a ratio >= 1 to tau_m
# so that m is always the fast gate (the current only depends on m * h, so the two could swap).
def segment_params(params, k):
    tau_m, ratio, m_inf, h_inf = params[2+4*k:6+4*k]
    return tau_m, tau_m * ratio, m_inf, h_inf

# All 10 segments are fitted together, with one shared e_r and g_bar.
# Segments at the same voltage share their gate parameters: seg_v[j-1] is the voltage index of segment j.
def compute_error_segment(params, data, past, seg_v, k_hold):
    e_r, g_bar = params[:2]
    _, _, m_0, h_0 = segment_params(params, k_hold)   # start from the holding steady state
    errors = []
    for j in range(1, 11):
        start = j*100-past
        stop = (j+1)*100
        tau_m, tau_h, m_inf, h_inf = segment_params(params, seg_v[j-1])
        current, m_0, h_0 = get_current_segment(e_r, g_bar, tau_m, tau_h, m_inf, h_inf, data[start:stop, 1], data[start:stop, 0], m_0, h_0)
        errors.append(current - data[start:stop, 2])
    return np.concatenate(errors)

# Whole-trace model: m_inf/h_inf are sigmoids of voltage, tau_m/tau_h are Gaussians of voltage
def get_current(e_r, g_bar, m_params, h_params, tau_m_params, tau_h_params, v, t):
    m = np.zeros(len(v))
    h = np.zeros(len(v))

    m_inf = sigmoid(*m_params, v)
    h_inf = sigmoid(*h_params, v)
    tau_m = np.maximum(gaussian(*tau_m_params, v), 1e-3)   # floor: a Gaussian goes to 0 far from its center
    tau_h = np.maximum(gaussian(*tau_h_params, v), 1e-3)

    m[0] = m_inf[0]   # the trace starts at the holding voltage, so start at steady state
    h[0] = h_inf[0]

    for i in range(1, len(t)):
        dt = t[i] - t[i-1]

        m[i] = gate_model(tau_m[i], m[i-1], m_inf[i], dt)
        h[i] = gate_model(tau_h[i], h[i-1], h_inf[i], dt)

    return g_bar * (v - e_r) * m * h

def compute_error(params, t, v, test_i):
    e_r, g_bar = params[:2]
    return get_current(e_r, g_bar, params[2:4], params[4:6], params[6:9], params[9:12], v, t) - test_i

def random_start():
    e_r   = rng.uniform(-100, 50)
    g_bar = 10 ** rng.uniform(-3, 0)    # log-uniform: spans orders of magnitude
    m_inf = [ 10 ** rng.uniform(-2, -0.5), rng.uniform(-80, 20)]    # steepness > 0: rises with voltage
    h_inf = [-10 ** rng.uniform(-2, -0.5), rng.uniform(-80, 20)]    # steepness < 0: falls with voltage
    tau_m = [10 ** rng.uniform(-1, 1), rng.uniform(-80, 20), 10 ** rng.uniform(2, 4)]
    tau_h = [10 ** rng.uniform(0, 2), rng.uniform(-80, 20), 10 ** rng.uniform(2, 4)]
    return [e_r, g_bar, *m_inf, *h_inf, *tau_m, *tau_h]

def gaussian(scale,center,width,v):
    return scale*np.exp(-((v-center)**2)/width)

def compute_error_tau(params,v,tau):
    scale,center,width = params
    return gaussian(scale,center,width,v) - tau # Gaussian formula

def sigmoid(steepness,center,v):
    return 1/(1+np.exp(-steepness*(v-center)))

def compute_error_x_inf(params,v,x_inf):
    steepness,center = params
    return sigmoid(steepness,center,v) - x_inf

def plot_x_inf(x_inf,v,name="x_inf"):
    fit = least_squares(compute_error_x_inf,args=(v,x_inf),x0=[1,0])

    voltages = [i for i in range(-100,21)]

    plt.figure()

    plt.subplot(1, 1, 1)
    plt.grid(True)
    plt.scatter(v, x_inf, linewidth=2, label="Segment fits")
    plt.plot(voltages,sigmoid(fit.x[0],fit.x[1],voltages), label="Sigmoid fit")
    plt.ylabel(name)
    plt.xlabel("Voltage (mV)")
    plt.legend()
    return fit.x

def plot_taus(v,tau_m,tau_h):

    fit_m = least_squares(compute_error_tau, args = (v,tau_m), x0 = [3,20,2000])
    fit_h = least_squares(compute_error_tau, args = (v,tau_h), x0 = [30,-40,2000])

    voltages = [i for i in range(-100,21)]

    plt.figure()

    plt.subplot(2, 1, 1)
    plt.grid(True)
    plt.scatter(v,tau_m, linewidth=2)
    plt.plot(voltages,gaussian(fit_m.x[0],fit_m.x[1],fit_m.x[2],voltages))
    plt.axis("tight")
    plt.ylabel("tau_m (msec)")

    plt.subplot(2, 1, 2)
    plt.grid(True)
    plt.scatter(v, tau_h, linewidth=2, label="Segment fits")
    plt.plot(voltages,gaussian(fit_h.x[0],fit_h.x[1],fit_h.x[2],voltages), label="Gaussian fit")
    plt.ylabel("tau_h (msec)")
    plt.xlabel("Voltage (mV)")
    plt.legend()
    return fit_m.x, fit_h.x

# Model comparison (lower is better). Assumes Gaussian noise, so the likelihood
# reduces to the residual sum of squares; constant terms are dropped.
def compute_aic(residuals, n_params):
    n = len(residuals)
    ss_res = np.sum(np.asarray(residuals) ** 2)
    return n * np.log(ss_res / n) + 2 * n_params

def compute_bic(residuals, n_params):
    n = len(residuals)
    ss_res = np.sum(np.asarray(residuals) ** 2)
    return n * np.log(ss_res / n) + n_params * np.log(n)

# Fit on the train_mask samples only, then score on the held-out rest. The gates
# carry state, so the full trace is always simulated and the mask picks which points count.
def fit_holdout(x0, t, v, test_i, train_mask):
    starts = [x0] + [random_start() for _ in range(n_starts - 1)]
    fits = [least_squares(lambda p: compute_error(p, t, v, test_i)[train_mask], x0=s, bounds=(lower, upper), max_nfev=40) for s in starts]
    fit = min(fits, key=lambda f: f.cost)   # best start, judged on the training points only
    res = compute_error(fit.x, t, v, test_i)
    r2 = lambda m: 1 - np.sum(res[m] ** 2) / np.sum((test_i[m] - np.mean(test_i[m])) ** 2)
    return r2(train_mask), r2(~train_mask), np.sqrt(np.mean(res[~train_mask] ** 2))

In [ ]:
# %% SECTION 3: Main function
def run_vclamp_model(folder: str, data: str = "CookLab1UnknownChannelExtra.txt", run_validation: bool = False):
    """
    Compute current from voltage-clamp input.

    Parameters
    ----------
    folder :
        Directory with input file for voltage-clamp model.
    data :
        Name of input file.
    # Please list any parameters added to the function.
    run_validation :
        If True, also run the train/test hold-out experiments (slow).

    Returns
    -------
    np.ndarray
        Output current computed by the voltage-clamp model.
    """
    # %% SECTION 3-a: Load provided input data.
    # NO MODIFICATIONS: This section should be kept as is.
    data = np.loadtxt(os.path.join(folder, data))
    v = data[:, 1]

    # i has the same shape as array v, add code in the next section to fill it
    i = np.empty_like(v)

    # %% SECTION 3-b: Compute i [to be filled by students]
    rng.bit_generator.state = np.random.default_rng(0).bit_generator.state   # reseed: every call uses the same random starts
    tau_m = []
    tau_h = []
    tau_v = []
    list_i = []
    m_inf = []
    h_inf = []
    m_0 = []
    h_0 = []
    past=1

    # One fit for all 10 segments (see compute_error_segment)
    seg_volts = [data[j*100+50, 1] for j in range(1, 11)]
    tau_v = sorted(set(seg_volts))
    seg_v = [tau_v.index(x) for x in seg_volts]
    k_hold = tau_v.index(data[0, 1])
    # [e_r, g_bar], then [tau_m, tau_h / tau_m, m_inf, h_inf] for each voltage
    lower_seg = [-150, 0]   + [0.01, 1,   0, 0] * len(tau_v)
    upper_seg = [ 150, 100] + [100,  1e4, 1, 1] * len(tau_v)
    fit = least_squares(compute_error_segment, args = (data, past, seg_v, k_hold), x0 = [-20, 0.01] + [1, 5, 0.5, 0.5] * len(tau_v), bounds = (lower_seg, upper_seg), max_nfev = 400)
    print("Fit succeeded:", fit.success)

    for k in range(len(tau_v)):
        seg = segment_params(fit.x, k)
        print(f"Fitted parameters for V={tau_v[k]}:", seg)
        tau_m.append(seg[0])
        tau_h.append(seg[1])
        m_inf.append(seg[2])
        h_inf.append(seg[3])

    for j in range(1,11):
        start = j*100-past
        stop = (j+1)*100
        t = data[start:stop, 0]
        v = data[start:stop, 1]

        if j == 1:
            m_0.append(m_inf[k_hold])
            h_0.append(h_inf[k_hold])
        seg = segment_params(fit.x, seg_v[j-1])
        current,m_last,h_last = get_current_segment(fit.x[0],fit.x[1],*seg,v,t,m_0[-1],h_0[-1])
        i = current.copy()
        m_0.append(m_last)
        h_0.append(h_last)
        list_i+=list(i)

    # Rescale the gates to peak at 1 and move the factor into g_bar (the current only depends on g_bar * m * h)
    e_r = fit.x[0]
    g_bar = fit.x[1] * max(m_inf) * max(h_inf)
    m_inf = np.array(m_inf) / max(m_inf)
    h_inf = np.array(h_inf) / max(h_inf)

    tau_m_fit, tau_h_fit = plot_taus(tau_v,tau_m,tau_h)
    m_fit = plot_x_inf(m_inf,tau_v,"m_inf")
    h_fit = plot_x_inf(h_inf,tau_v,"h_inf")

    t = data[:, 0]
    v = data[:, 1]
    test_i = data[:,2]

    fits = []
    for k in range(n_starts):
        if k == 0:
            x0 = np.clip([e_r, g_bar, *m_fit, *h_fit, *tau_m_fit, *tau_h_fit], lower, upper)   # the segment fits' curves
        else:
            x0 = random_start()
        fit = least_squares(compute_error, x0=x0, args=(t, v, test_i), bounds=(lower, upper), max_nfev=40,)
        fits.append(fit)
        print(f"start {k}: cost = {fit.cost:.4f}, success = {fit.success}, params = {fit.x}")
    best = min(fits, key=lambda f: f.cost)
    best = least_squares(compute_error, x0=best.x, args=(t, v, test_i), bounds=(lower, upper), max_nfev=200)   # let the best start converge
    print("Best parameters:", best.x)
    fit = best
    i = get_current(fit.x[0], fit.x[1], fit.x[2:4], fit.x[4:6], fit.x[6:9], fit.x[9:12], v, t)

    ss_res = np.sum((test_i - i) ** 2)
    ss_tot = np.sum((test_i - np.mean(test_i)) ** 2)
    r_squared = 1 - (ss_res / ss_tot)
    print("R-squared:", r_squared)

    print("AIC:", compute_aic(test_i - i, len(fit.x)))
    print("BIC:", compute_bic(test_i - i, len(fit.x)))

    print("Fitted parameters:", fit.x)
    print("Fit succeeded:", fit.success)

    # Train/test experiments on the whole-trace model
    if run_validation:
        x0 = np.clip([e_r, g_bar, *m_fit, *h_fit, *tau_m_fit, *tau_h_fit], lower, upper)
        idx = np.arange(len(t))
        for frac in [0.05, 0.10, 0.20, 0.30, 0.50]:   # hold out the last frac of the trace
            train_r2, test_r2, test_rmse = fit_holdout(x0, t, v, test_i, idx < len(t) * (1 - frac))
            print(f"tail {frac:.0%} held out: train R2 = {train_r2:.3f}, test R2 = {test_r2:.3f}, test RMSE = {test_rmse:.4f}")
        for j in [1, 3, 5, 7, 9]:                     # hold out one voltage step ("hump") at a time
            train_r2, test_r2, test_rmse = fit_holdout(x0, t, v, test_i, (idx < j*100) | (idx >= (j+1)*100))
            print(f"V = {v[j*100]:.0f} mV step held out: train R2 = {train_r2:.3f}, test R2 = {test_r2:.3f}, test RMSE = {test_rmse:.4f}")

    plt.figure()

    plt.subplot(2, 1, 1)
    plt.grid(True)
    plt.plot(data[:, 0], data[:, 1], linewidth=2)
    plt.axis("tight")
    plt.ylabel("V step (mV)")

    plt.subplot(2, 1, 2)
    plt.grid(True)
    plt.plot(data[:, 0], data[:, 2], label="Measured current")
    plt.plot(data[:, 0], i, label="Whole-trace model", color="k")
    t = []
    for j in range(1,11):
        start = j*100-past
        stop = (j+1)*100

        # if j%2==1:
        t += list(data[start:stop, 0])
    plt.scatter(t, list_i, label="Segment model",color="r")
    plt.xlabel("Time (msec)")
    plt.ylabel("Current (mA)")
    plt.legend()

    # %% SECTION 3-c: Assert validity of output and return it.
    # NO MODIFICATIONS: This section should be kept as is.
    assert len(i.shape) == 1, "Currrent array should be 1-dimensional"
    return i

In [ ]:
# %% SECTION 4: Function calls
if __name__ == "__main__":
    # Replace this with the path to the folder where provided input .txt file is!
    folder = "data"

    # Run function on the voltage-clamp data
    i = run_vclamp_model(folder)